In [3]:
import pandas as pd
import plotly.express as px

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Loo UrbanStyle näidisandmed (kui CSV puudub)
data = {
    'customer_id': [1001, 1002, 1003, 1001, 1002, 1004, 1003, 1001, 1005, 1004,
                    1002, 1003, 1005, 1001, 1006, 1004, 1002, 1007, 1003, 1005],
    'sale_date': ['2024-01-15', '2024-01-16', '2024-02-01', '2024-02-20', '2024-03-01',
                   '2024-03-05', '2024-03-15', '2024-04-10', '2024-04-12', '2024-04-20',
                   '2024-05-01', '2024-05-10', '2024-05-15', '2024-06-01', '2024-06-05',
                   '2024-06-10', '2024-06-20', '2024-07-01', '2024-07-05', '2024-07-10'],
    'total_price': [89.99, 45.50, 120.00, 67.30, 55.00, 210.00, 33.50, 145.00, 78.00, 92.00,
                     160.00, 44.00, 88.50, 230.00, 37.00, 175.00, 110.00, 65.00, 95.00, 125.00],
    'city': ['Tallinn', 'Tartu', 'Tallinn', 'Tallinn', 'Tartu', 'Pärnu', 'Tallinn', 'Tallinn',
             'Tartu', 'Pärnu', 'Tartu', 'Tallinn', 'Tartu', 'Tallinn', 'Pärnu', 'Pärnu',
             'Tartu', 'Tallinn', 'Tallinn', 'Tartu'],
    'product_category': ['Dresses', 'Tops', 'Denim', 'Accessories', 'Tops', 'Denim', 'Tops',
                         'Dresses', 'Denim', 'Accessories', 'Dresses', 'Tops', 'Denim',
                         'Dresses', 'Accessories', 'Denim', 'Tops', 'Accessories', 'Dresses', 'Denim']
}
df = pd.DataFrame(data)

# Kasuta olemasolevat df-i või loo uuesti (vt Osa 1A)
# Veendu, et sale_date on datetime tüüpi
df['sale_date'] = pd.to_datetime(df['sale_date'])

# Viitekuupäev (viimase tellimuse järgmine päev)
today = pd.to_datetime('2024-08-01')
print("Andmestiku periood:", df['sale_date'].min(), "kuni", df['sale_date'].max())
print("Viitekuupäev:", today)

Andmestiku periood: 2024-01-15 00:00:00 kuni 2024-07-10 00:00:00
Viitekuupäev: 2024-08-01 00:00:00


In [4]:
# Recency: päevi viimasest ostust
recency = df.groupby('customer_id')['sale_date'].max().reset_index()
recency.columns = ['customer_id', 'last_purchase']
recency['recency_days'] = (today - recency['last_purchase']).dt.days

# Frequency: ostude arv
frequency = df.groupby('customer_id').size().reset_index(name='frequency')

# Monetary: kogukulutus
monetary = df.groupby('customer_id')['total_price'].sum().reset_index()
monetary.columns = ['customer_id', 'monetary']

# Liida kokku
rfm = recency[['customer_id', 'recency_days']].merge(
    frequency, on='customer_id'
).merge(
    monetary, on='customer_id'
)

print(rfm)

   customer_id  recency_days  frequency  monetary
0         1001            61          4    532.29
1         1002            42          4    370.50
2         1003            27          4    292.50
3         1004            52          3    477.00
4         1005            22          3    291.50
5         1006            57          1     37.00
6         1007            31          1     65.00


In [5]:
# Lihtsustatud skooride määramine (täida lüngad!)
# Recency: väiksem = parem (hiljuti ostis), seega pööra ümber
rfm['R_score'] = pd.qcut(rfm['recency_days'], q=3, labels=[3, 2, 1]).astype(int)

# Frequency: suurem = parem
rfm['F_score'] = pd.qcut(
    rfm['frequency'].rank(method='first'), q=3, labels=[1, 2, 3]
).astype(int)

# Monetary: suurem = parem
rfm['M_score'] = pd.qcut(rfm['monetary'], q=3, labels=[1, 2, 3]).astype(int)

# Koguskoor
rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

# Segmenteerimine
def assign_segment(score):
    if score >= 8:
        return 'VIP Champions'
    elif score >= 6:
        return 'Loyal Customers'
    elif score >= 4:
        return 'Potential Loyalists'
    else:
        return 'At Risk'

rfm['segment'] = rfm['RFM_score'].apply(assign_segment)
print(rfm.sort_values('RFM_score', ascending=False))

   customer_id  recency_days  frequency  monetary  R_score  F_score  M_score  \
2         1003            27          4    292.50        3        3        2   
1         1002            42          4    370.50        2        3        2   
0         1001            61          4    532.29        1        2        3   
3         1004            52          3    477.00        2        1        3   
4         1005            22          3    291.50        3        2        1   
6         1007            31          1     65.00        3        1        1   
5         1006            57          1     37.00        1        1        1   

   RFM_score              segment  
2          8        VIP Champions  
1          7      Loyal Customers  
0          6      Loyal Customers  
3          6      Loyal Customers  
4          6      Loyal Customers  
6          5  Potential Loyalists  
5          3              At Risk  


In [6]:
# Graafik 1: Segmentide jaotus
segment_counts = rfm['segment'].value_counts().reset_index()
fig1 = px.bar(
    segment_counts,
    x='segment',
    y='count',
    title='UrbanStyle: Kliendisegmentide jaotus (RFM)',
    labels={'segment': 'Segment', 'count': 'Klientide arv'},
    color='segment'
)
fig1.show()

# Graafik 2: Hajuvusdiagramm — Recency vs Monetary
fig2 = px.scatter(
    rfm,
    x='recency_days',
    y='monetary',
    color='segment',
    size='frequency',
    hover_data=['customer_id'],
    title='UrbanStyle: Recency vs Monetary (RFM)',
    labels={
        'recency_days': 'Päevi viimasest ostust',
        'monetary': 'Kogukulutus (EUR)'
    }
)
fig2.show()